# Challenge 1 - Tic Tac Toe

In this lab you will perform deep learning analysis on a dataset of playing [Tic Tac Toe](https://en.wikipedia.org/wiki/Tic-tac-toe).

There are 9 grids in Tic Tac Toe that are coded as the following picture shows:

![Tic Tac Toe Grids](tttboard.jpg)

In the first 9 columns of the dataset you can find which marks (`x` or `o`) exist in the grids. If there is no mark in a certain grid, it is labeled as `b`. The last column is `class` which tells you whether Player X (who always moves first in Tic Tac Toe) wins in this configuration. Note that when `class` has the value `False`, it means either Player O wins the game or it ends up as a draw.

Follow the steps suggested below to conduct a neural network analysis using Tensorflow and Keras. You will build a deep learning model to predict whether Player X wins the game or not.

## Step 1: Data Engineering

This dataset is almost in the ready-to-use state so you do not need to worry about missing values and so on. Still, some simple data engineering is needed.

1. Read `tic-tac-toe.csv` into a dataframe.
1. Inspect the dataset. Determine if the dataset is reliable by eyeballing the data.
1. Convert the categorical values to numeric in all columns.
1. Separate the inputs and output.
1. Normalize the input data.

In [3]:
# your code here
import pandas as pd
import numpy as np

In [5]:
df = pd.read_csv("/content/tic-tac-toe.csv")
print(f"Shape: {df.shape}")
print(df.head())

Shape: (958, 10)
  TL TM TR ML MM MR BL BM BR  class
0  x  x  x  x  o  o  x  o  o   True
1  x  x  x  x  o  o  o  x  o   True
2  x  x  x  x  o  o  o  o  x   True
3  x  x  x  x  o  o  o  b  b   True
4  x  x  x  x  o  o  b  o  b   True


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 958 entries, 0 to 957
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   TL      958 non-null    object
 1   TM      958 non-null    object
 2   TR      958 non-null    object
 3   ML      958 non-null    object
 4   MM      958 non-null    object
 5   MR      958 non-null    object
 6   BL      958 non-null    object
 7   BM      958 non-null    object
 8   BR      958 non-null    object
 9   class   958 non-null    bool  
dtypes: bool(1), object(9)
memory usage: 68.4+ KB


In [6]:
df.isna().sum()

,0
TL,0
TM,0
TR,0
ML,0
MM,0
MR,0
BL,0
BM,0
BR,0
class,0


In [7]:
df.isnull().sum()

,0
TL,0
TM,0
TR,0
ML,0
MM,0
MR,0
BL,0
BM,0
BR,0
class,0


In [10]:
df.duplicated().sum()

np.int64(0)

In [38]:
df['class'].value_counts()

,count
class,
True,626
False,332


The dataframe is composed by 958 rows and 10 columns.
There are no missing or duplicated entries.
All columns but class are objects, so they will be converted into numerical.
Only True or False values are in the 'class' column.

In [45]:
# Create a copy of the df
df_num = df.copy()

In [46]:
# Convert x, o, and b into numerical values
cell_map = {'x': 1, 'o': -1, 'b': 0}
for col in df.columns[:-1]:
    df_num[col] = df_num[col].map(cell_map)

df_num.head()

,TL,TM,TR,ML,MM,MR,BL,BM,BR,class
0,1,1,1,1,-1,-1,1,-1,-1,True
1,1,1,1,1,-1,-1,-1,1,-1,True
2,1,1,1,1,-1,-1,-1,-1,1,True
3,1,1,1,1,-1,-1,-1,0,0,True
4,1,1,1,1,-1,-1,0,-1,0,True


In [47]:
df_num.dtypes

,0
TL,int64
TM,int64
TR,int64
ML,int64
MM,int64
MR,int64
BL,int64
BM,int64
BR,int64
class,bool


In [48]:
# Enconde the class column to numerical
df_num['class'] = df_num['class'].map({True: 1, False: 0})
print(df_num.head())
print(df_num['class'].value_counts())

   TL  TM  TR  ML  MM  MR  BL  BM  BR  class
0   1   1   1   1  -1  -1   1  -1  -1      1
1   1   1   1   1  -1  -1  -1   1  -1      1
2   1   1   1   1  -1  -1  -1  -1   1      1
3   1   1   1   1  -1  -1  -1   0   0      1
4   1   1   1   1  -1  -1   0  -1   0      1
class
1    626
0    332
Name: count, dtype: int64


All values were enconded into numerical.
The value counts of the 'class' column are in accordance with the counts from True/False.

In [52]:
# Separate the inputs (X) and output (y)
X = df_num.drop('class', axis=1)
y = df_num['class']

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

X shape: (958, 9)
y shape: (958,)


Data is already normalized, considering it has values between -1 and 1.

## Step 2: Build Neural Network

To build the neural network, you can refer to your own codes you wrote while following the [Deep Learning with Python, TensorFlow, and Keras tutorial](https://www.youtube.com/watch?v=wQ8BIBpya2k) in the lesson. It's pretty similar to what you will be doing in this lab.

1. Split the training and test data.
1. Create a `Sequential` model.
1. Add several layers to your model. Make sure you use ReLU as the activation function for the middle layers. Use Softmax for the output layer because each output has a single lable and all the label probabilities add up to 1.
1. Compile the model using `adam` as the optimizer and `sparse_categorical_crossentropy` as the loss function. For metrics, use `accuracy` for now.
1. Fit the training data.
1. Evaluate your neural network model with the test data.
1. Save your model as `tic-tac-toe.model`.

In [56]:
# your code here
import tensorflow as tf
import tensorflow.keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input, Flatten
from tensorflow.keras.optimizers import RMSprop, Adam
from sklearn.model_selection import train_test_split

In [55]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (766, 9)
X_test shape: (192, 9)
y_train shape: (766,)
y_test shape: (192,)


In [58]:
766*9

6894

In [86]:
# Build the model
model = Sequential()
model.add(Input(shape=(9,)))
model.add(Flatten())
model.add(Dense(32, activation='relu', input_shape=(9,)))
model.add(Dropout(0.2))
model.add(Dense(2, activation='softmax')) #Two outputs

model.summary()
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_2 (Flatten)             │ (None, 9)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 32)             │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 2)              │            66 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 386 (1.51 KB)

 Trainable params: 386 (1.51 KB)

 Non-trainable params: 0 (0.00 B)

In [87]:
# Fit the model
model.fit(X_train, y_train,
          batch_size=8,
          epochs=20,
          verbose=1,
          validation_data=(X_test, y_test))

Epoch 1/20
96/96 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.5574 - loss: 0.7315 - val_accuracy: 0.6406 - val_loss: 0.6358
Epoch 2/20
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6593 - loss: 0.6144 - val_accuracy: 0.6979 - val_loss: 0.5912
Epoch 3/20
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6893 - loss: 0.5958 - val_accuracy: 0.7344 - val_loss: 0.5647
Epoch 4/20
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7037 - loss: 0.5562 - val_accuracy: 0.7396 - val_loss: 0.5471
Epoch 5/20
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7219 - loss: 0.5471 - val_accuracy: 0.7500 - val_loss: 0.5295
Epoch 6/20
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7389 - loss: 0.5143 - val_accuracy: 0.7656 - val_loss: 0.5138
Epoch 7/20
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7650 - loss: 0.4840 - val_accuracy: 0.7812 - val_loss: 0.4945
Epoch 8/20
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7650 - loss: 0.4744 - val_accuracy: 0.7812 - val_loss:

In [88]:
# Evaluate the model
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print("Test loss:", test_loss)
print("Test accuracy:", test_accuracy)

Test loss: 0.2163156419992447
Test accuracy: 0.9479166865348816


The model achieved a high test accuracy (0.95), but a test loss of 0.22, which should be improved.

In [89]:
# Save the model
model.save("tic-tac-toe.keras")

In [90]:
from google.colab import files
files.download("tic-tac-toe.keras")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Step 3: Make Predictions

Now load your saved model and use it to make predictions on a few random rows in the test dataset. Check if the predictions are correct.

In [91]:
# your code here
# Load the model
model=tf.keras.models.load_model('tic-tac-toe.keras')

In [92]:
# Select 10 random rows from test set
random_indices = np.random.choice(len(X_test), size=10, replace=False)
X_sample = X_test.iloc[random_indices]
y_sample = y_test.iloc[random_indices]

In [93]:
predictions = model.predict(X_sample)
predicted_classes = np.argmax(predictions, axis=1)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step


In [94]:
for i in range(len(X_sample)):
    print("Row:", random_indices[i])
    print("Predicted:", predicted_classes[i])
    print("Actual:   ", y_sample.iloc[i])
    print("Correct?: ", predicted_classes[i] == y_sample.iloc[i])
    print("-" * 30)

Row: 127
Predicted: 1
Actual:    1
Correct?:  True
------------------------------
Row: 148
Predicted: 1
Actual:    1
Correct?:  True
------------------------------
Row: 76
Predicted: 1
Actual:    1
Correct?:  True
------------------------------
Row: 42
Predicted: 0
Actual:    0
Correct?:  True
------------------------------
Row: 1
Predicted: 1
Actual:    1
Correct?:  True
------------------------------
Row: 129
Predicted: 1
Actual:    1
Correct?:  True
------------------------------
Row: 104
Predicted: 1
Actual:    1
Correct?:  True
------------------------------
Row: 186
Predicted: 1
Actual:    0
Correct?:  False
------------------------------
Row: 60
Predicted: 1
Actual:    1
Correct?:  True
------------------------------
Row: 99
Predicted: 1
Actual:    1
Correct?:  True
------------------------------


The model succeeded in predicting the correct outcome for 9 of the test samples.

## Step 4: Improve Your Model

Did your model achieve low loss (<0.1) and high accuracy (>0.95)? If not, try to improve your model.

But how? There are so many things you can play with in Tensorflow and in the next challenge you'll learn about these things. But in this challenge, let's just do a few things to see if they will help.

* Add more layers to your model. If the data are complex you need more layers. But don't use more layers than you need. If adding more layers does not improve the model performance you don't need additional layers.
* Adjust the learning rate when you compile the model. This means you will create a custom `tf.keras.optimizers.Adam` instance where you specify the learning rate you want. Then pass the instance to `model.compile` as the optimizer.
    * `tf.keras.optimizers.Adam` [reference](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/Adam).
    * Don't worry if you don't understand what the learning rate does. You'll learn about it in the next challenge.
* Adjust the number of epochs when you fit the training data to the model. Your model performance continues to improve as you train more epochs. But eventually it will reach the ceiling and the performance will stay the same.

In [97]:
# your code here
# Add another layer to the model
model = Sequential()
model.add(Input(shape=(9,)))
model.add(Flatten())
model.add(Dense(32, activation='relu', input_shape=(9,)))
model.add(Dropout(0.2))
model.add(Dense(32, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(2, activation='softmax'))

model.summary()
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Fit the model
model.fit(X_train, y_train,
          batch_size=8,
          epochs=20,
          verbose=1,
          validation_data=(X_test, y_test))

# Evaluate the model
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print("Test loss:", test_loss)
print("Test accuracy:", test_accuracy)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_5 (Flatten)             │ (None, 9)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 32)             │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 2)              │            66 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,442 (5.63 KB)

 Trainable params: 1,442 (5.63 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
96/96 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.5496 - loss: 0.7116 - val_accuracy: 0.6823 - val_loss: 0.6247
Epoch 2/20
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6997 - loss: 0.6103 - val_accuracy: 0.7604 - val_loss: 0.5828
Epoch 3/20
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7128 - loss: 0.5861 - val_accuracy: 0.7708 - val_loss: 0.5540
Epoch 4/20
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7167 - loss: 0.5578 - val_accuracy: 0.7812 - val_loss: 0.5320
Epoch 5/20
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7480 - loss: 0.5222 - val_accuracy: 0.7812 - val_loss: 0.4987
Epoch 6/20
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7768 - loss: 0.5031 - val_accuracy: 0.8229 - val_loss: 0.4674
Epoch 7/20
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7950 - loss: 0.4670 - val_accuracy: 0.8281 - val_loss: 0.4209
Epoch 8/20
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8303 - loss: 0.4100 - val_accuracy: 0.8698 - val_loss:

Adding another layer improved the model's performannce, by increasing the test accuracy to 0.97 and reducing the test loss to 0.11.
Another layer will be added to try to improved the test loss further.

In [99]:
# Add another layer to the model
model = Sequential()
model.add(Input(shape=(9,)))
model.add(Flatten())
model.add(Dense(32, activation='relu', input_shape=(9,)))
model.add(Dropout(0.2))
model.add(Dense(32, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(16, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(2, activation='softmax'))

model.summary()
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Fit the model
model.fit(X_train, y_train,
          batch_size=8,
          epochs=20,
          verbose=1,
          validation_data=(X_test, y_test))

# Evaluate the model
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print("Test loss:", test_loss)
print("Test accuracy:", test_accuracy)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_7 (Flatten)             │ (None, 9)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, 32)             │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_16 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_24 (Dense)                │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_17 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_25 (Dense)                │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_18 (Dropout)            │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_26 (Dense)                │ (None, 2)              │            34 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,938 (7.57 KB)

 Trainable params: 1,938 (7.57 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
96/96 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.5783 - loss: 0.6781 - val_accuracy: 0.6458 - val_loss: 0.6287
Epoch 2/20
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6723 - loss: 0.6052 - val_accuracy: 0.6823 - val_loss: 0.5848
Epoch 3/20
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7154 - loss: 0.5701 - val_accuracy: 0.7760 - val_loss: 0.5328
Epoch 4/20
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7324 - loss: 0.5557 - val_accuracy: 0.7917 - val_loss: 0.5000
Epoch 5/20
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7794 - loss: 0.5031 - val_accuracy: 0.8125 - val_loss: 0.4564
Epoch 6/20
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7977 - loss: 0.4672 - val_accuracy: 0.8646 - val_loss: 0.4031
Epoch 7/20
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8211 - loss: 0.4221 - val_accuracy: 0.8750 - val_loss: 0.3485
Epoch 8/20
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8342 - loss: 0.3929 - val_accuracy: 0.8854 - val_loss:

Adding another layer improved slightly the test accuracy but not the test loss.
Reducing the learning rate will be evaluated.

In [100]:
# Reduce learning rate
model = Sequential()
model.add(Input(shape=(9,)))
model.add(Flatten())
model.add(Dense(32, activation='relu', input_shape=(9,)))
model.add(Dropout(0.2))
model.add(Dense(32, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(16, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(2, activation='softmax'))

model.summary()
model.compile(optimizer=Adam(learning_rate=0.0001),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Fit the model
model.fit(X_train, y_train,
          batch_size=8,
          epochs=20,
          verbose=1,
          validation_data=(X_test, y_test))

# Evaluate the model
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print("Test loss:", test_loss)
print("Test accuracy:", test_accuracy)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_8 (Flatten)             │ (None, 9)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_27 (Dense)                │ (None, 32)             │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_19 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_28 (Dense)                │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_20 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_29 (Dense)                │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_21 (Dropout)            │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_30 (Dense)                │ (None, 2)              │            34 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,938 (7.57 KB)

 Trainable params: 1,938 (7.57 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
96/96 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.5157 - loss: 0.6983 - val_accuracy: 0.6094 - val_loss: 0.6632
Epoch 2/20
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5379 - loss: 0.6902 - val_accuracy: 0.6406 - val_loss: 0.6489
Epoch 3/20
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5836 - loss: 0.6784 - val_accuracy: 0.6562 - val_loss: 0.6387
Epoch 4/20
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6084 - loss: 0.6612 - val_accuracy: 0.6562 - val_loss: 0.6298
Epoch 5/20
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6240 - loss: 0.6582 - val_accuracy: 0.6771 - val_loss: 0.6227
Epoch 6/20
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6188 - loss: 0.6551 - val_accuracy: 0.6771 - val_loss: 0.6156
Epoch 7/20
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6292 - loss: 0.6475 - val_accuracy: 0.6927 - val_loss: 0.6090
Epoch 8/20
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6397 - loss: 0.6470 - val_accuracy: 0.6979 - val_loss:

Reducing the learning rate reduced the model's performance by increasing the test loss and decreasing the test accuracy.

Increasing the learning rate

In [101]:
# Increase learning rate
model = Sequential()
model.add(Input(shape=(9,)))
model.add(Flatten())
model.add(Dense(32, activation='relu', input_shape=(9,)))
model.add(Dropout(0.2))
model.add(Dense(32, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(16, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(2, activation='softmax'))

model.summary()
model.compile(optimizer=Adam(learning_rate=0.01),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Fit the model
model.fit(X_train, y_train,
          batch_size=8,
          epochs=20,
          verbose=1,
          validation_data=(X_test, y_test))

# Evaluate the model
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print("Test loss:", test_loss)
print("Test accuracy:", test_accuracy)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_9 (Flatten)             │ (None, 9)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_31 (Dense)                │ (None, 32)             │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_22 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_32 (Dense)                │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_23 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_33 (Dense)                │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_24 (Dropout)            │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_34 (Dense)                │ (None, 2)              │            34 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,938 (7.57 KB)

 Trainable params: 1,938 (7.57 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
96/96 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.6984 - loss: 0.5967 - val_accuracy: 0.7656 - val_loss: 0.4936
Epoch 2/20
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7742 - loss: 0.4770 - val_accuracy: 0.7917 - val_loss: 0.4222
Epoch 3/20
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8655 - loss: 0.3192 - val_accuracy: 0.9427 - val_loss: 0.2218
Epoch 4/20
96/96 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9164 - loss: 0.2093 - val_accuracy: 0.9688 - val_loss: 0.1603
Epoch 5/20
96/96 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9530 - loss: 0.1807 - val_accuracy: 0.9740 - val_loss: 0.1459
Epoch 6/20
96/96 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9582 - loss: 0.1451 - val_accuracy: 0.9740 - val_loss: 0.1281
Epoch 7/20
96/96 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9634 - loss: 0.1244 - val_accuracy: 0.9740 - val_loss: 0.1183
Epoch 8/20
96/96 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9608 - loss: 0.1140 - val_accuracy: 0.9688 - val_loss:

Increasing the learning rate increased the test loss.

Increasing the number of epoch will be evaluated.

In [103]:
# Increase number of epochs
model = Sequential()
model.add(Input(shape=(9,)))
model.add(Flatten())
model.add(Dense(32, activation='relu', input_shape=(9,)))
model.add(Dropout(0.2))
model.add(Dense(32, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(16, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(2, activation='softmax'))

model.summary()
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Fit the model
model.fit(X_train, y_train,
          batch_size=8,
          epochs=40,
          verbose=1,
          validation_data=(X_test, y_test))

# Evaluate the model
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print("Test loss:", test_loss)
print("Test accuracy:", test_accuracy)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_12"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_11 (Flatten)            │ (None, 9)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_39 (Dense)                │ (None, 32)             │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_28 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_40 (Dense)                │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_29 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_41 (Dense)                │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_30 (Dropout)            │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_42 (Dense)                │ (None, 2)              │            34 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,938 (7.57 KB)

 Trainable params: 1,938 (7.57 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/40
96/96 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.6266 - loss: 0.6565 - val_accuracy: 0.7083 - val_loss: 0.5995
Epoch 2/40
96/96 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7076 - loss: 0.5877 - val_accuracy: 0.7344 - val_loss: 0.5398
Epoch 3/40
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7180 - loss: 0.5518 - val_accuracy: 0.7917 - val_loss: 0.4785
Epoch 4/40
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7611 - loss: 0.4981 - val_accuracy: 0.8177 - val_loss: 0.4312
Epoch 5/40
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7728 - loss: 0.4701 - val_accuracy: 0.8490 - val_loss: 0.3778
Epoch 6/40
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8068 - loss: 0.4116 - val_accuracy: 0.8854 - val_loss: 0.3169
Epoch 7/40
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8316 - loss: 0.3735 - val_accuracy: 0.9219 - val_loss: 0.2609
Epoch 8/40
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8773 - loss: 0.3064 - val_accuracy: 0.9479 - val_loss:

Increasing the number of epochs improved the model's performance, maintaining high test accuracy (0.97) and reducing the test loss to 0.08.

**Which approach(es) did you find helpful to improve your model performance?**

In [ ]:
# your answer here

Increasing the number of layers and the number of epochs improved the model's performance.
Adjusting the learning rate did not improved the model.